# Import data and library

In [ ]:
# handle with data
import pandas as pd
import numpy as np

# visulize
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# normalize the data
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# model using
import xgboost as xgb
from xgboost import plot_importance

# evaluation metrics
from sklearn.metrics import mean_squared_log_error

In [ ]:
# import data
train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv')
test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv')
oil = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv')
stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')

In [ ]:
# mrege oil to train and test
train = train.merge(oil, on='date', how='left')
test = test.merge(oil, on='date', how='left')
train = train.merge(stores, on='store_nbr', how='left')
test = test.merge(stores, on='store_nbr', how='left')

# Check the data

In [ ]:
train

In [ ]:
test

In [ ]:
train.isnull().sum()

In [ ]:
train.info()

In [ ]:
train.nunique()

In [ ]:
train.describe()

In [ ]:
print(train.columns)
print(test.columns)

# Plot to check the data

In [ ]:
print(train['family'].nunique()*train['store_nbr'].nunique())
print(train['date'].nunique())
print(test.shape)

In [ ]:
train_plot = train.groupby('family')['store_nbr'].count()
fig = px.bar(train_plot,y='store_nbr',title="Number of family with store_nbr")
fig.show()

In [ ]:
for col in ['onpromotion','sales']:
    train_plot = train.groupby(['store_nbr','family'])[col].mean().reset_index()
    fig = px.line(train_plot, x='store_nbr', y=col, color='family', title=f'Mean {col} on each store_nbr and family')
    fig.show()

In [ ]:
for col in ['onpromotion','sales']:
    plot = train.groupby('date')[col].mean().reset_index()
    fig = px.line(plot, x='date', y=col, title=f'Total {col} per day')
    fig.show()

In [ ]:
for col in ['sales', 'onpromotion']:  
    for t in ['store_nbr', 'family']:
        plot = train.groupby(t)[col].mean()
        fig = px.bar(plot, y=col, title=f'Mean {col} in {t}')
        fig.show()

- onpromotion start form 0 for a while then increasing
- sales slightly increase
- store_nbr and family are both effect to onpromotion and sales
- each of values in store_nbr and family has the same number

# Date time

date is now object type. So, I will change it to datetime to do something.

In [ ]:
train['date'] = pd.to_datetime(train['date'], format='%Y-%m-%d')
test['date'] = pd.to_datetime(test['date'], format='%Y-%m-%d')
train.date.dtype

In [ ]:
# create year day_of_week and month columns from date.
train['year'] = train.date.dt.year
train['day_of_week'] = train.date.dt.dayofweek
train['month'] = train.date.dt.month

test['year'] = test.date.dt.year
test['day_of_week'] = test.date.dt.dayofweek
test['month'] = test.date.dt.month

In [ ]:
for col in ['onpromotion', 'sales']:
    for t in ['day_of_week', 'month','year']:
        time = train.groupby(t)[col].mean()
        fig = px.bar(time, y=col, title=f'Mean {col} each {t}')
        fig.show()

- year increase both of onpromotion and sales
- month onpromotion on midyer and endyear sales increase on 12th month
- day onpromotion no trend but sales has a bit slightly increase on midmonth and endmonth
- day_of_week onpromotion hightest in 2nd and 4th and sales hightest on 5th and 6th

# Preparing the data

## Merged train and test

In [ ]:
data = pd.concat([train, test])

In [ ]:
data.nunique()

## Handle with null

In [ ]:
data.isnull().sum()

In [ ]:
data['dcoilwtico'] = (data['dcoilwtico'].fillna(method='ffill') + data['dcoilwtico'].fillna(method='bfill')) / 2

In [ ]:
data.fillna(0, inplace=True)

In [ ]:
data.isnull().sum()

## Seperate id, date and sale columns

In [ ]:
# create new variable to contain id and date columns for both train and test.
train_uniq = train[['id', 'date']]
test_uniq = test[['id', 'date']]

# create new variable to contain sales column.
train_sales = train[['sales']]

# drop id, date and sales.
data = data.drop(columns=['id', 'date', 'sales'])

## One hot encoding with dummy code

In [ ]:
data = pd.get_dummies(data, drop_first=True)

## Normalization with min max scaling

In [ ]:
scaler = MinMaxScaler()
data = pd.DataFrame(scaler.fit_transform(data), columns=data.columns)
train_sales = pd.DataFrame(scaler.fit_transform(train_sales), columns=train_sales.columns)

In [ ]:
data.describe()

## Seperate train and test back

In [ ]:
train_prep = data[:1782*1684]
train_sales = train_sales[:1782*1684]
# 1782 from each date has 33 families and 54 stores

In [ ]:
test_prep = data[1684*1782:]

# XGBoost

## Predict real target data

In [ ]:
XGB_model = xgb.XGBRegressor(objective="count:poisson",booster='gbtree', n_estimators=200, max_depth=100, learning_rate=0.4, random_state=123)

In [ ]:
XGB_model.fit(train_prep, train_sales)

In [ ]:
pred_train = XGB_model.predict(train_prep)
pred_test = XGB_model.predict(test_prep)

In [ ]:
pred_train = pd.DataFrame(pred_train)
pred_train = scaler.inverse_transform(pred_train)
pred_train = np.array(pred_train)

train_sales = pd.DataFrame(train_sales)
train_sales = scaler.inverse_transform(train_sales)
train_sales = np.array(train_sales)

pred_test = pd.DataFrame(pred_test)
pred_test = scaler.inverse_transform(pred_test)
pred_test = np.array(pred_test)

In [ ]:
rmsle = mean_squared_log_error(train_sales, pred_train, squared=False)
print(f'RMSLE on real training set: {rmsle}')

## Plot to see the result

In [ ]:
pred_train = pd.DataFrame({'pred_train': pred_train.ravel()})
pred_test = pd.DataFrame({'pred_train': pred_test.ravel()})

In [ ]:
train['sales'].plot(kind='line', figsize=(15,5))

In [ ]:
pred_train['pred_train'].plot(kind='line', figsize=(15,5))

In [ ]:
pred_test['pred_train'].plot(kind='line', figsize=(15,5))

# Create submission files

In [ ]:
pred_test

In [ ]:
submit = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/sample_submission.csv')

In [ ]:
submit['sales'] = pred_test

In [ ]:
submit

In [ ]:
submit.to_csv('submission.csv', index=False)